#### Step 3: LLM Model Evaluation

In this notebook, you'll deploy the Meta Llama 2 7B model and evaluate it's text generation capabilities and domain knowledge. You'll use the SageMaker Python SDK for Foundation Models and deploy the model for inference. 

The Llama 2 7B Foundation model performs the task of text generation. It takes a text string as input and predicts next words in the sequence. 

#### Set Up
There are some initial steps required for setup. If you receive warnings after running these cells, you can ignore them as they won't impact the code running in the notebook. Run the cell below to ensure you're using the latest version of the SageMaker Python client library. Restart the Kernel after you run this cell. 

In [ ]:
import sys

# Reset SageMaker SDK packages, then use the stable 2.x JumpStart API path.
# After this cell finishes, restart the kernel before running the cells below.
!{sys.executable} -m pip install -U pip --quiet
!{sys.executable} -m pip uninstall -y \
    sagemaker \
    sagemaker-core \
    sagemaker-train \
    sagemaker-serve \
    sagemaker-mlops \
    --quiet
!{sys.executable} -m pip install --no-cache-dir \
    "sagemaker>=2.231.0,<3" \
    datasets \
    ipywidgets \
    --quiet


***! Restart the notebook kernel now after running the above cell and before you run any cells below !*** 

To deploy the model on Amazon SageMaker, we need to setup and authenticate the use of AWS services. Yo'll uuse the execution role associated with the current notebook instance as the AWS account role with SageMaker access. Validate your role is the SageMaker IAM role you created for the project by running the next cell. 

In [ ]:
import boto3
import sagemaker
from sagemaker.session import get_execution_role

boto_session = boto3.Session()
aws_region = boto_session.region_name
sess = sagemaker.Session(boto_session=boto_session)

try:
    role = get_execution_role(sess)
except Exception:
    identity = boto_session.client("sts").get_caller_identity()
    caller_arn = identity["Arn"]
    account_id = identity["Account"]
    if ":assumed-role/" in caller_arn:
        role_name = caller_arn.split(":assumed-role/", 1)[1].split("/", 1)[0]
        role = f"arn:aws:iam::{account_id}:role/{role_name}"
    else:
        role = caller_arn

print(role)
print(aws_region)
print(sess)


## 2. Select Text Generation Model Meta Llama 2 7B
Run the next cell to set variables that contain the values of the name of the model we want to load and the version of the model .

In [ ]:
model_id = "meta-textgeneration-llama-2-7b"


Running the next cell deploys the model.
This Python code uses the SageMaker SDK V3 JumpStart and ModelBuilder APIs.

1. Create a `JumpStartConfig` with the model ID.

2. Create a `ModelBuilder` from that JumpStart configuration and set the instance type.

3. Build the model and deploy it to a SageMaker endpoint.

The returned `endpoint` object is used to invoke the deployed model and later delete the endpoint resources.

**The next cell will take some time to run.** It is deploying a large language model, and that takes time. You'll see progress while it is being deployed. Please be patient until the model is deployed, then continue running the next cells.


In [ ]:
import uuid
import boto3
from sagemaker.jumpstart.model import JumpStartModel

name_suffix = str(uuid.uuid4())[:8]
endpoint_name = f"llama2-eval-{name_suffix}"
instance_type = "ml.g5.2xlarge"

# Pin the JumpStart model version so the notebook is not surprised by a future
# model signature change. If this version is unavailable in your region, remove
# model_version and rerun.
model = JumpStartModel(
    model_id=model_id,
    model_version="5.1.0",
    role=role,
    sagemaker_session=sess,
)

try:
    predictor = model.deploy(
        initial_instance_count=1,
        instance_type=instance_type,
        endpoint_name=endpoint_name,
        accept_eula=True,
    )
except Exception as e:
    print(f"Deployment failed for endpoint: {endpoint_name}")
    print(e)
    try:
        details = boto3.client("sagemaker").describe_endpoint(EndpointName=endpoint_name)
        print("Status:", details.get("EndpointStatus"))
        print("FailureReason:", details.get("FailureReason", ""))
    except Exception as describe_error:
        print("Could not describe failed endpoint:", describe_error)
    raise


In [ ]:
import boto3

# Check the endpoint created in the previous cell. Do not hard-code an endpoint
# name from an older run, and do not delete it until after evaluation.
sm_client = boto3.client("sagemaker")
details = sm_client.describe_endpoint(EndpointName=endpoint_name)
endpoint_config_name = details["EndpointConfigName"]

print("EndpointName:", endpoint_name)
print("Status:", details.get("EndpointStatus"))
print("EndpointConfigName:", endpoint_config_name)
print("FailureReason:", details.get("FailureReason", ""))


#### Invoke the endpoint, query and parse response
The next step is to invoke the model endpoint, send a query to the endpoint, and receive a response from the model. 

Running the next cell defines a function that will be used to parse and print the response from the model. 

In [ ]:
def print_response(payload, response):
    if isinstance(response, list) and response:
        text = response[0].get("generation") or response[0].get("generated_text")
    elif isinstance(response, dict):
        text = response.get("generation") or response.get("generated_text") or response
    else:
        text = response

    print(payload["inputs"])
    print(f"> {text}")
    print("\n==================================\n")


The model takes a text string as input and predicts next words in the sequence, the input we send it is the prompt. 

The prompt we send the model should relate to the domain we'd like to fine-tune the model on.  This way we'll identify the model's domain knowledge before it's fine-tuned, and then we can run the same prompts on the fine-tuned model.   

**Replace "inputs"** in the next cell with the input to send the model based on the domain you've chosen. 

**For financial domain:**

  "inputs": "Replace with sentence below"  
- "The  investment  tests  performed  indicate"
- "the  relative  volume  for  the  long  out  of  the  money  options, indicates"
- "The  results  for  the  short  in  the  money  options"
- "The  results  are  encouraging  for  aggressive  investors"

**For medical domain:** 

  "inputs": "Replace with sentence below" 
- "Myeloid neoplasms and acute leukemias derive from"
- "Genomic characterization is essential for"
- "Certain germline disorders may be associated with"
- "In contrast to targeted approaches, genome-wide sequencing"

**For IT domain:** 

  "inputs": "Replace with sentence below" 
- "Traditional approaches to data management such as"
- "A second important aspect of ubiquitous computing environments is"
- "because ubiquitous computing is intended to" 
- "outline the key aspects of ubiquitous computing from a data management perspective."

In [ ]:
from sagemaker.serializers import JSONSerializer
from sagemaker.deserializers import JSONDeserializer

payload = {
    "inputs": "The investment tests performed indicate",
    "parameters": {
        "max_new_tokens": 64,
        "top_p": 0.9,
        "temperature": 0.6,
        "return_full_text": False,
    },
}

predictor.serializer = JSONSerializer()
predictor.deserializer = JSONDeserializer()

try:
    try:
        response = predictor.predict(payload, custom_attributes="accept_eula=true")
    except TypeError:
        response = predictor.predict(payload)

    print_response(payload, response)
except Exception as e:
    print(e)


The prompt is related to the domain you want to fine-tune your model on. You will see the outputs from the model without fine-tuning are limited in providing insightful or relevant content.

**Use the output from this notebook to fill out the "model evaluation" section of the project documentation report**

Take a screenshot of this file with the cell output for your project documentation report. Download it with cell output by making sure you used Save on the notebook before downloading 

**After you've filled out the report, run the cells below to delete the endpoint** 

`IF YOU FAIL TO RUN THE CELLS BELOW YOU WILL RUN OUT OF BUDGET TO COMPLETE THE PROJECT`

In [ ]:
import boto3

sm_client = boto3.client("sagemaker")

try:
    predictor.delete_endpoint(delete_endpoint_config=True)
    print(f"Deleted endpoint and endpoint config: {endpoint_name}")
except Exception as e:
    print(f"Endpoint cleanup skipped/failed: {e}")

model_name = getattr(model, "name", None)
if model_name:
    try:
        sm_client.delete_model(ModelName=model_name)
        print(f"Deleted model: {model_name}")
    except Exception as e:
        print(f"Model cleanup skipped/failed: {e}")


Verify your endpoint was deleted by visiting the SageMaker dashboard and choosing `endpoints` under 'Inference' in the left navigation menu.  If you see your endpoint still there, choose the endpoint, and then under "Actions" select **Delete**